# Implementing the libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Abstract

In this notebook, we study the [Knight's Tour](https://en.wikipedia.org/wiki/Knight%27s_tour) problem, which asks whether a knight can move across an $n \times n$ chessboard so that every square is visited exactly once. We present the problem from a mathematical perspective by modeling the chessboard as a graph, where each square is represented by a vertex and each legal knight move by an edge. In this setting, an open knight's tour corresponds to a Hamiltonian path, while a closed knight's tour corresponds to a Hamiltonian cycle.

To investigate the problem computationally, we implement and compare three different approaches: a backtracking algorithm, Warnsdorff's heuristic, and a neural-network-based method inspired by the formulation presented on the Knight's Tour Wikipedia page. The methods are evaluated primarily in terms of runtime, while also considering their practical reliability and scalability as the board size increases.

Our main focus is on the open knight's tour problem. In the final part of the notebook, we apply the most effective method to construct closed knight's tours whenever the board dimensions permit such a solution.

# 2. Methods

## 2.1. How the knight moves

A knight moves in an $L$-shape: two squares in one direction and one square in the perpendicular direction. If a square is represented by coordinates $(x,y)$, then the set of all legal knight moves is

$$\mathcal{M}=\{(\pm 1,\pm 2),(\pm 2,\pm 1)\}.$$

Thus, from a square $(x,y)$, the knight may move to any square of the form

$$(x,y) + m, \qquad m \in \mathcal{M},$$

provided that the destination square still lies within the board.

For an $n \times n$ chessboard, we define the set of board squares by

$$V_n = \{(i,j)\mid 1 \leq i,j \leq n\}.$$

Two squares $(i,j)$ and $(k,\ell)$ are adjacent if and only if

$$(k-i,\ell-j)\in \mathcal{M}.$$

This allows us to define the **knight graph**

$$G_n = (V_n,E_n),$$

where each vertex corresponds to a square of the board, and each edge corresponds to a legal knight move.

In [ ]:
KNIGHT_STEPS = [(2,1),(1,2),(-1,2),(-2,1),(-2,-1),(-1,-2),(1,-2),(2,-1)]

## 2.2. Open and closed knight's tours

A knight's tour is a sequence of distinct squares

$$v_1,v_2,\dots,v_{n^2}, \qquad v_k \in V_n,$$

such that each consecutive pair is connected by a legal knight move:

$$\{v_k,v_{k+1}\}\in E_n, \qquad k=1,2,\dots,n^2-1.$$

### Open knight's tour

An **open knight's tour** is a sequence

$$v_1,v_2,\dots,v_{n^2}$$

such that

$$v_i \neq v_j \quad \text{for } i\neq j,$$

and

$$\{v_k,v_{k+1}\}\in E_n\quad \text{for all } k=1,\dots,n^2-1.$$

Since every square is visited exactly once, an open knight's tour is exactly a **Hamiltonian path** in the knight graph $G_n$.

### Closed knight's tour

A **closed knight's tour** is an open knight's tour with the additional property that the final square is also a legal knight move away from the starting square:

$$\{v_{n^2},v_1\}\in E_n.$$

Therefore, a closed knight's tour is exactly a **Hamiltonian cycle** in the knight graph $G_n$.


In [2]:
def all_squares(n):
    """
    Return all squares of an n x n chessboard using 0-based indexing.

    Parameters
    ----------
    n : int - Board dimension.

    Returns
    -------
    list[tuple[int, int]] - List of coordinates (i, j) for all board squares.
    """
    return [(row, column) for row in range(n) for column in range(n)]

In [3]:
def legal_moves(n, square):
    """
    Return all legal knight moves from a given square on an n x n board.

    Parameters
    ----------
    n : int - Board dimension.
    square : tuple[int, int] - Current square (row, column).

    Returns
    -------
    list[tuple[int, int]] - List of reachable squares by one legal knight move.
    """
    row, column = square
    moves = []
    for drow, dcolumn in KNIGHT_STEPS:
        res_row, res_column = row + drow, column + dcolumn
        if 0 <= res_row < n and 0 <= res_column < n:
            moves.append((res_row, res_column))
    return moves

In [4]:
def knight_graph(n):
    """
    Construct the knight graph of an n x n board.

    Each square is a vertex, and edges connect squares that differ by a legal knight move.

    Parameters
    ----------
    n : int - Board dimension.

    Returns
    -------
    dict[tuple[int, int], list[tuple[int, int]]] - Adjacency dictionary of the knight graph.
    """
    return {tile: legal_moves(n, tile) for tile in all_squares(n)}

In [5]:
def is_knight_move(a, b):
    """
    Check whether moving from square a to square b is a legal knight move.

    Parameters
    ----------
    a : tuple[int, int] - Starting square.
    b : tuple[int, int] - Destination square.

    Returns
    -------
    bool - True if b is reachable from a by one knight move, otherwise False.
    """
    drow = abs(a[0] - b[0])
    dcolumn = abs(a[1] - b[1])
    return sorted((drow, dcolumn)) == [1, 2]

In [6]:
def is_valid_tour(tour, n, closed = False):
    """
    Check whether a sequence of squares is a valid knight's tour on an n x n board.

    A valid tour must:
    - contain exactly n^2 squares,
    - visit each square exactly once,
    - stay inside the board,
    - move by legal knight moves between consecutive squares.

    If closed = True, the final square must also be a legal knight move away
    from the starting square.

    Parameters
    ----------
    tour : list[tuple[int, int]] or None - Candidate tour.
    n : int - Board dimension.
    closed : bool, default=False - Whether to require a closed tour.

    Returns
    -------
    bool - True if the sequence is a valid tour, otherwise False.
    """
    if tour is None or len(tour) != n * n:
        return False
    if len(set(tour)) != n * n:
        return False
    if any(not (0 <= row < n and 0 <= column < n) for row, column in tour):
        return False
    if any(not is_knight_move(tour[i], tour[i + 1]) for i in range(len(tour) - 1)):
        return False
    if closed and not is_knight_move(tour[-1], tour[0]):
        return False
    return True


In [7]:
def open_tour_exists_square(n):
    """
    Return whether an open knight's tour exists on an n x n square board.

    Classical existence result:
    - True for n = 1
    - True for all n >= 5
    - False for n = 2, 3, 4

    Parameters
    ----------
    n : int - Board dimension.

    Returns
    -------
    bool - True if an open tour exists, otherwise False.
    """
    return n == 1 or n >= 5

In [8]:
def closed_tour_exists_square(n):
    """
    Return whether a closed knight's tour exists on an n x n square board.

    Classical existence result:
    - A closed tour exists iff n >= 6 and n is even.

    Parameters
    ----------
    n : int - Board dimension.

    Returns
    -------
    bool - True if a closed tour exists, otherwise False.
    """
    return n >= 6 and n % 2 == 0

In [9]:
def tour_to_matrix(tour, n):
    """
    Convert a knight's tour into an n x n matrix of visit times.

    The entry M[row, column] equals k if the square (row, column) is visited at step k.

    Parameters
    ----------
    tour : list[tuple[int, int]] - Knight's tour represented as an ordered list of squares.
    n : int - Board dimension.

    Returns
    -------
    numpy.ndarray - n x n integer matrix whose entries record visit order.
    """
    M = np.zeros((n, n), dtype=int)
    for k, (row, column) in enumerate(tour, start=1):
        M[row, column] = k
    return M

Before proceeding with the implementation of the main algorithms we'll define two key functions that would help us visualize the knight's tours.

In [11]:
def plot_tour(tour, n, ax=None, title=None, annotate=None):
    """
    Plot a knight's tour on an n x n chessboard.

    The board is drawn as a square grid with alternating light shading,
    and the tour is displayed as a polyline connecting the visited squares
    in order. Optionally, each step of the tour can be annotated with its
    visit number.

    Parameters
    ----------
    tour : list[tuple[int, int]] - Ordered sequence of board squares representing the knight's tour.
                                   Each square is given as (row, column) using 0-based indexing.
    n : int - Board dimension.
    ax : matplotlib.axes.Axes, optional - Existing matplotlib axes on which to draw the tour. If None,
                                          a new figure and axes are created.
    title : str, optional - Title of the plot.
    annotate : bool, optional - If True, label each visited square with its step number.
                                If None, annotation is enabled automatically for boards of size n <= 8.

    Returns
    -------
    matplotlib.axes.Axes - The axes containing the plot.

    Raises
    ------
    ValueError - If the input tour is None.
    """
    if tour is None:
        raise ValueError("Tour must not be None")
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))
    if annotate is None:
        annotate = n <= 8

    # Board grid
    for i in range(n + 1):
        ax.plot([0, n], [i, i], lw=1, color='black')
        ax.plot([i, i], [0, n], lw=1, color='black')

    # Alternating board shading
    for r in range(n):
        for c in range(n):
            if (r + c) % 2 == 0:
                ax.add_patch(plt.Rectangle((c, n - 1 - r), 1, 1,
                                           color='lightgray', alpha=0.25))

    xs = [c + 0.5 for r, c in tour]
    ys = [n - 1 - r + 0.5 for r, c in tour]
    ax.plot(xs, ys, marker='o', lw=1.5, color='black')

    if annotate:
        for k, (x, y) in enumerate(zip(xs, ys), start=1):
            ax.text(x, y, str(k), ha='center', va='center',
                    fontsize=8, color='black')

    ax.set_xlim(0, n)
    ax.set_ylim(0, n)
    ax.set_aspect('equal')
    ax.set_xticks(range(n + 1))
    ax.set_yticks(range(n + 1))
    ax.tick_params(colors='black')

    if title:
        ax.set_title(title, color='black')

    return ax

## 2.3. Backtracking algorithm

The first approach considered in this notebook is the **backtracking algorithm**, which is an exact recursive search procedure on the knight graph.

Suppose that after $k$ steps we have constructed a partial tour

$$P_k = (v_1,v_2,\dots,v_k),$$

with visited set

$$S_k = \{v_1,v_2,\dots,v_k\}.$$

At step $k$, the set of admissible next moves consists of all unvisited neighbors of the current square $v_k$:

$$A(v_k,S_k)=\{u\in N(v_k)\mid u\notin S_k\},$$

where $N(v_k)$ denotes the set of graph neighbors of $v_k$.

The recursive backtracking rule may be expressed as

$$
\mathrm{Tour}(v_k,S_k)=
\begin{cases}
\text{success}, & |S_k| = n^2, \\[6pt]
\displaystyle \bigvee_{u\in A(v_k,S_k)} \mathrm{Tour}(u,S_k\cup\{u\}), & |S_k| < n^2.
\end{cases}
$$

In words, the algorithm works as follows:

1. Start from an initial square $v_1$.
2. Choose one legal unvisited neighboring square.
3. Add it to the current path.
4. Continue recursively.
5. If a dead end is reached before all $n^2$ squares are visited, return to the previous step and try another move.

This method is guaranteed to find a solution if one exists, because it systematically explores the search tree of all feasible knight-move sequences. However, the number of possible paths grows very quickly with the board size, so plain backtracking becomes computationally expensive for larger boards.

## 2.4. Warnsdorff's heuristic

The second method is **Warnsdorff's heuristic**, a classical greedy strategy for constructing knight's tours efficiently.

The main idea is to move the knight at each step to the square from which the number of onward legal moves is as small as possible.

For a candidate next square $u \in A(v_k,S_k)$, define its onward degree by

$$d_{S_k}(u) = \left|\left\{w \in N(u)\mid w \notin S_k \cup \{u\}\right\}\right|.$$

Warnsdorff's rule then chooses the next square according to

$$v_{k+1}=\arg\min_{u\in A(v_k,S_k)} d_{S_k}(u).$$

Thus, among all currently legal unvisited moves, the algorithm selects the one that leaves the fewest future options.

The intuition behind this heuristic is that squares with low accessibility should be visited early, before they become unreachable later in the tour. Although this method is not exhaustive like backtracking, it performs remarkably well in practice and often finds tours very quickly even on large boards.

If several candidate squares have the same onward degree, a tie-breaking rule must be used. In practice, ties may be broken arbitrarily, randomly, or by an additional secondary criterion.

## 5. Neural-network method

The third approach is a **neural-network-based method**, inspired by the formulation presented in the Knight's Tour Wikipedia article.

In this framework, the problem is reformulated as a dynamical system. Each legal knight move is associated with a neuron, and each neuron can be either active or inactive. The network evolves iteratively until it reaches a stable state.

Let $N_{i,j}$ denote the neuron associated with a legal move from square $i$ to square $j$. For each neuron we define:

- $U_t(N_{i,j})$: its internal state at time $t$,
- $V_t(N_{i,j}) \in \{0,1\}$: its output at time $t$,
- $G(N_{i,j})$: the set of neurons interacting with $N_{i,j}$.

The state update rule is

$$U_{t+1}(N_{i,j})=U_t(N_{i,j})+ 2-\sum_{N\in G(N_{i,j})} V_t(N),$$

and the output update rule is

$$V_{t+1}(N_{i,j}) =
\begin{cases}
1, & U_{t+1}(N_{i,j}) > 3, \\[6pt]
0, & U_{t+1}(N_{i,j}) < 0, \\[6pt]
V_t(N_{i,j}), & \text{otherwise.}
\end{cases}
$$

The goal of this process is for the network to converge to a configuration in which the active neurons encode a valid knight's tour.

If convergence is achieved, the active set of neurons may represent either:

1. a valid knight's tour, or
2. several disconnected cycles or partial structures.

For this reason, the neural-network approach is best viewed as an experimental optimization-based method rather than a guaranteed constructive algorithm. Its mathematical interest lies in transforming a discrete combinatorial search problem into an iterative dynamical system.